In [6]:
pip install Levenshtein pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 5.7 MB/s  0:00:00 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [Levenshtein]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Загрузка переменных окружения

In [13]:
from dotenv import load_dotenv
import os
from pathlib import Path
load_dotenv()
PATH_FINTOC_2022 = Path(os.getenv('PATH_FINTOC_2022'))

In [2]:
FINTOC_EN = PATH_FINTOC_2022/'data'/'en'
FINTOC_EN_PDF =FINTOC_EN/'pdf'
FINTOC_EN_ANNOTS = FINTOC_EN/'annots'
FINTOC_EN_PRED = FINTOC_EN/'preds'

JSON_EXTENSION = "fintoc4.json"

In [3]:
if not FINTOC_EN_PRED.exists():
    FINTOC_EN_PRED.mkdir()
FILE_PATHS = list(FINTOC_EN_PDF.iterdir())[:10]

In [4]:
import json

In [5]:
import pager
from pager.pager_pipe_line import pdf2region_row

from pager.doc_model import MinerPDFModel
from pager.doc_model import LogicTreeModel
from pager.doc_model.converters import PDF2LogicTree

from typing import Dict

def tree2fintoc_json(doc_tree) -> Dict:
    fintoc_json = []
    for id_node, node in doc_tree['nodes']['regions'].items():
        if node['label'] == 'header':
            reg = {
                "text":node['text'],
                "id": id_node,
                "page": node['metainfo']['page'] + 1, # В PageR специально оставил нумерацию с нуля (поскольку номер определяется не порядком, а пишится на странице и это еще не реализовано)
                "depth": node['header_level']
            }
            fintoc_json.append(reg)
    return fintoc_json
    

pdf_reader = MinerPDFModel({"page_model": pdf2region_row})
pdf2tree = PDF2LogicTree()
tree = LogicTreeModel()

In [6]:
N = len(FILE_PATHS)
for i, file_path in enumerate(FILE_PATHS):    
    pdf_reader.read_from_file(file_path)
    pdf_reader.extract()
    pdf2tree.convert(pdf_reader, tree)
    
    
    doc_tree = tree.to_dict()
    pred_path = FINTOC_EN_PRED/(file_path.name+'.'+JSON_EXTENSION)
    # print(tree.document.md)
    
    rez = tree2fintoc_json(doc_tree)
    with open(pred_path, 'w') as f:
        json.dump(rez, f)

    print(f'{i+1}/{N} ({(i+1)/N*100:.2f}%)')

/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch_geometric/nn/conv/tag_conv.py:102: UserWarning: Converting sparse tensor to CSR format for more efficient processing. Consider converting your sparse tensor to CSR format beforehand to avoid repeated conversion (got 'torch.sparse_coo')
  return spmm(adj_t, x, reduce=self.aggr)
/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch_geometric/utils/_spmm.py:71: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  src = src.to_sparse_csr()
/home/daniil/project/PageR/env/lib/python3.12/site-packages/torch/nn/modules/module.py:1775: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


1/10 (10.00%)
2/10 (20.00%)
3/10 (30.00%)
4/10 (40.00%)


Cannot set gray non-stroke color because /'P0' is an invalid float value


5/10 (50.00%)
6/10 (60.00%)
7/10 (70.00%)
8/10 (80.00%)
9/10 (90.00%)
10/10 (100.00%)


# Запуск теста

In [9]:
import subprocess
import pandas as pd

In [10]:
subprocess.run(['python', 'metric.py', '--gt_folder', FINTOC_EN_ANNOTS, '--submission_folder', FINTOC_EN_PRED])

CompletedProcess(args=['python', 'metric.py', '--gt_folder', PosixPath('/home/dataset/fintoc2022/data/en/annots'), '--submission_folder', PosixPath('/home/dataset/fintoc2022/data/en/preds')], returncode=0)

# Вывод результатов

### TD

In [11]:
df = pd.read_csv('td_report.csv', delimiter='\t')
df[-2:-1][['Prec', 'Rec', 'F1']]

,Prec,Rec,F1
36,0.5480608384487492,0.42292489487466345,0.46266263377904887


### ToC

In [12]:
df = pd.read_csv('toc_report.csv', delimiter='\t')
tbl = df[-4:-3]
tbl[['Inex08-P', 'Inex08-R', 'Inex08-F1', 'Inex08-Title acc', 'Inex08-Level acc']]

,Inex08-P,Inex08-R,Inex08-F1,Inex08-Title acc,Inex08-Level acc
40,30.4,22.3,25.0,36.5,12.1
